# 06 · Validation report — metrics, AQI confusion matrix, Taylor, residual map

**BAH 2026 PS3 · the consolidated, score-relevant validation page.**

PS3 scores Objective-1 on **RMSE / R / MAE** and rewards leakage-free rigour. This notebook
assembles the validation deliverables from `aqi_india.validation`: the core **metrics table**
(RMSE/MAE/MBE/R/R²/NMB/NME/IOA + RMA slope), the **AQI-category confusion matrix** (with
hazardous-class recall), a **Taylor diagram**, the **1:1 hexbin**, and a per-station **residual
map** — all from honest leave-station-out out-of-fold predictions.

In [ ]:
import sys, pathlib
# Make the src/ layout importable when running from the notebooks/ folder
# without an editable install. If aqi_india is already installed this is a no-op.
_repo = pathlib.Path.cwd()
for _ in range(4):
    if (_repo / 'src' / 'aqi_india').is_dir():
        sys.path.insert(0, str(_repo / 'src'))
        break
    _repo = _repo.parent
import aqi_india
print('aqi_india', aqi_india.__version__)

## 1. Rebuild the Objective-1 OOF predictions

We reproduce the leave-station-out out-of-fold predictions for the surface pollutants (the same
construction as notebook 03), since honest validation must score each station with a model that
never saw it. We gap-fill, build the feature matrix, and collect OOF pairs per pollutant.

In [ ]:
import numpy as np, pandas as pd
from aqi_india.sim import synthetic as sim
from aqi_india.fusion.gapfill import fill_gaps
from aqi_india.features.feature_matrix import build_feature_matrix, feature_columns
from aqi_india.models.lightgbm_model import LGBMParams, make_single_pollutant_factory, SURFACE_POLLUTANTS
from aqi_india.validation.cv import leave_station_out

SEED = 42
grid = sim.make_grid('2023-10-01', n_days=60, res=0.25, seed=SEED)
fires = sim.make_fires(grid, season='oct_nov', seed=SEED)
grid = sim.inject_fire_hcho(grid, fires)
stations = sim.make_stations(grid, n=120, seed=SEED)
grid = fill_gaps(grid, method='auto')
matrix = build_feature_matrix(grid, stations, fires)
feat_cols = feature_columns(matrix)
params = LGBMParams(n_estimators=400, learning_rate=0.05, seed=SEED)
print('matrix', matrix.shape, '| predictors', len(feat_cols))

In [ ]:
def oof_predictions(pollutant):
    sub = matrix.dropna(subset=[pollutant, *feat_cols]).reset_index(drop=True)
    X = sub[feat_cols]; y = sub[pollutant].to_numpy(float)
    groups = sub['station_id'].to_numpy()
    factory = make_single_pollutant_factory(pollutant, params=params, feature_names=feat_cols)
    yt, yp, idx = [], [], []
    for tr, te in leave_station_out(X, groups, n_splits=5):
        est = factory(); est.fit(X.iloc[tr], y[tr])
        yt.append(y[te]); yp.append(np.asarray(est.predict(X.iloc[te]))); idx.append(te)
    return (np.concatenate(yt), np.concatenate(yp), sub.iloc[np.concatenate(idx)].reset_index(drop=True))

pollutants = [p for p in SURFACE_POLLUTANTS if p in matrix]
oof = {p: oof_predictions(p) for p in pollutants}
{p: int(oof[p][0].size) for p in pollutants}

## 2. The core metrics table (all pollutants)

`metrics_table(y_true, y_pred)` returns the full NaN-safe metric set. We tabulate every
pollutant; the **RMA slope** column exposes any compression of high-concentration episodes that
R² alone hides.

In [ ]:
from aqi_india.validation.metrics import metrics_table

rows = {p: metrics_table(oof[p][0], oof[p][1]) for p in pollutants}
metrics_df = pd.DataFrame(rows).T[['n', 'rmse', 'mae', 'mbe', 'r', 'r2', 'nmb', 'nme', 'ioa', 'rma_slope']]
metrics_df.round(3)

## 3. AQI-category confusion matrix

Beyond regression error, public-health messaging needs the **right CPCB band** — and especially
**recall on the hazardous classes** (Poor/Very Poor/Severe). We compute observed vs predicted
AQI per station-day from the OOF surface concentrations (assembling the per-pollutant OOF back
onto a common station-day index), then build the confusion matrix with
`aqi_india.validation.confusion.aqi_category_confusion`.

In [ ]:
from aqi_india.aqi.naqi import compute_aqi
from aqi_india.validation.confusion import aqi_category_confusion, HAZARDOUS_CATEGORIES

# Assemble OOF predicted + observed surface concentrations onto a shared station-day key.
key = ['station_id', 'time']
obs_tbl = matrix[key].copy()
pred_tbl = matrix[key].copy()
for p in pollutants:
    yt, yp, sub = oof[p]
    k = sub[key].copy(); k[p + '_obs'] = yt; k[p + '_pred'] = yp
    obs_tbl = obs_tbl.merge(k[[*key, p + '_obs']], on=key, how='left')
    pred_tbl = pred_tbl.merge(k[[*key, p + '_pred']], on=key, how='left')

obs_conc = obs_tbl.rename(columns={p + '_obs': p for p in pollutants})
pred_conc = pred_tbl.rename(columns={p + '_pred': p for p in pollutants})
aqi_obs = compute_aqi(obs_conc[pollutants])['aqi'].to_numpy()
aqi_pred = compute_aqi(pred_conc[pollutants])['aqi'].to_numpy()
valid = np.isfinite(aqi_obs) & np.isfinite(aqi_pred)
print('AQI pairs scored:', int(valid.sum()))

conf = aqi_category_confusion(aqi_obs[valid], aqi_pred[valid], normalize='true')
print('overall AQI-band accuracy : %.3f' % conf['accuracy'])
print('hazardous-class recall    : %.3f  (%s)' % (conf['hazardous_recall'], ', '.join(HAZARDOUS_CATEGORIES)))

In [ ]:
import matplotlib.pyplot as plt
labels = conf['labels']; M = np.asarray(conf['matrix'], dtype=float)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(M, cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label='row-normalised (recall)')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel('predicted band'); ax.set_ylabel('true band')
ax.set_title('AQI-category confusion matrix (row-normalised)')
for i in range(len(labels)):
    for j in range(len(labels)):
        if M[i, j] > 0:
            ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                    color='white' if M[i, j] > 0.5 else 'black', fontsize=8)
plt.tight_layout(); plt.show()

## 4. Taylor diagram (all pollutants, normalised)

One marker per pollutant: angle = correlation, radius = std normalised by the observed std, so
the reference (perfect) point sits at radius 1 on the x-axis.

In [ ]:
from aqi_india.validation.plots import taylor_diagram

stats = []
for p in pollutants:
    yt, yp, _ = oof[p]
    s_obs = float(np.std(yt))
    if s_obs > 0:
        stats.append({'name': p, 'std': float(np.std(yp)) / s_obs, 'r': metrics_df.loc[p, 'r']})
taylor_diagram(stats, ref_std=1.0, title='Taylor diagram — all surface pollutants (normalised)')
plt.show()

## 5. PM2.5 residual map over India

Where is the model biased? The per-station residual (pred − obs) bubble map reveals spatial
structure that scalar metrics hide.

In [ ]:
from aqi_india.validation.plots import one_to_one_hexbin, residual_map

yt, yp, sub = oof['pm25']
one_to_one_hexbin(yt, yp, units='ug/m3', title='PM2.5 — predicted vs observed (OOF)')
plt.show()
residual_map(sub[['lat', 'lon']], yp - yt, title='PM2.5 residuals (pred - obs) over India')
plt.show()

## Summary

The validation page consolidates the Objective-1 scoring story: a full per-pollutant metrics
table (RMSE/R/MAE + bias/agreement + RMA slope), an AQI-category confusion matrix that surfaces
hazardous-class recall, a Taylor diagram, the 1:1 hexbin and a residual map — all from
leave-station-out out-of-fold predictions so the numbers are leakage-free and reviewer-credible.
In a real run these same calls consume the trained-model OOF instead of the synthetic-fit demo.